# FINAL TAPER — six rewrites only

**Tuesday 6 October · 60 minutes maximum · closed book**

This is not a paper and not a topic-learning session. Rewrite the six known failure points from a blank cell, run the named tests, then stop. A part is unfinished until its required output is visible.

## Rules
- Work in order. Stop the whole notebook at 60 minutes even if unfinished.
- Do not open the boilerplates or previous attempts while writing.
- Do not use `sorted()` or `list.sort()` in T4.
- Keep each function’s named return exactly as specified.
- Run the checkpoints only after all attempts.

## T1 — hash find must probe and compare the key (10 min)

Use a global `hash_table` of 100 slots. Records are `[integer_key, string_data]`.

- `calculate_hash(key)` returns `key % 100`.
- `insert(key, data)` uses linear probing with wraparound. It returns `True` after insertion and `False` only after all 100 slots have been checked.
- `read_data(filename)` reads `records.txt` and calls `insert` for every row.
- `find_record(key)` starts at the hash, probes with wraparound, compares the stored key, returns only the string data, and returns `False` if an empty slot or 100 checked slots proves a miss.

**Required output:** after loading `records.txt`, print `find_record(475)` (must be `law`) and `find_record(9999)` (must be `False`).

In [2]:
# T1 — YOUR ATTEMPT
# identifiers: hash_table, calculate_hash, insert, read_data, find_record

global hash_table
hash_table = [None for _ in range(100)]

def calculate_hash(key):
    return key % 100

def insert(key,data):
    index = calculate_hash(key)
    if None not in hash_table:
        return False
    while index < 100 and hash_table[index] is not None:
        index += 1
        if index >= 100:
            index = 0
    hash_table[index] = [key,data]
    return True

def read_data(filename):
    with open(filename) as file:
        for row in file:
            row = row.strip().split(' ')
            insert(int(row[0]),row[1])
            
def find_record(key):
    index = calculate_hash(key)
    counter = 0
    while index < 100 and hash_table[index] is not None and hash_table[index][0] != key:
        counter += 1
        index += 1
        if index >= 100:
            index = 0
        if counter == 100:
            return False
    if hash_table[index] is None:
        return False
    return hash_table[index][1]

read_data('records.txt')
print(find_record(475))
print(find_record(9999))
    

law
False


## T2 — date expansion + SQL availability (12 min)

Database contract used by the checkpoint:

- `Villa(villaID INTEGER PRIMARY KEY, villaName TEXT)`
- `Villa_Booking(villaID INTEGER, date TEXT)`
- Dolphin is booked on `10-Apr` and `11-Apr`.

Write:

- `requested_dates(month, date, day_count)`: use a 2025 `datetime.date`; return consecutive strings in `%d-%b` format. Compute each as `start + timedelta(days=i)`.
- `availability(conn, villa_name, month, date, day_count)`: use parameterised SQL to fetch this villa’s occupied dates, convert rows to strings, and return `(available, unavailable)`.

**Required output:** Dolphin, Apr, 8, 4 must return available `['08-Apr', '09-Apr']` and unavailable `['10-Apr', '11-Apr']`.

In [3]:
# T2 — YOUR ATTEMPT
# identifiers: requested_dates, availability

import sqlite3, datetime

conn = sqlite3.connect('final_taper.db')
conn.row_factory = sqlite3.Row
cur = conn.cursor()

def requested_dates(month, date, daycount):
    output = []
    d = datetime.date(2025,datetime.datetime.strptime(month,'%b').month,date)
    for n in range(daycount):
        output.append(d.strftime('%d-%b'))
        d += datetime.timedelta(days=1)
    return output

def availability(conn,villa_name,month,date,day_count):
    cur = conn.cursor()
    cur.execute("""
    SELECT Villa_Booking.date
    FROM Villa_Booking INNER JOIN Villa ON Villa.villaID = Villa_Booking.villaID
    WHERE Villa.villaName = ?
    """,(villa_name,))
    booked = set()
    for row in cur.fetchall():
        booked.add(row['date'])
    tobook = set(requested_dates(month,date,day_count))
    unavailable = booked & tobook
    available = tobook - booked
    return list(available),list(unavailable)

print(availability(conn,'Dolphin','Apr',8,4))
conn.close()

(['08-Apr', '09-Apr'], ['10-Apr', '11-Apr'])


## T3 — write both named files and close them (6 min)

`write_html(name, text)` writes `<!doctype html>` followed by `text` to `name + '.html'`. Use `with open(..., 'w')` and return the completed filename.

Call it twice:

- `write_html('demo_one', '<html><body>one</body></html>')`
- `write_html('demo_two', '<html><body>two</body></html>')`

Print both filenames and both file sizes. Both sizes must be greater than 0.

In [4]:
# T3 — YOUR ATTEMPT
# identifier: write_html

def write_html(name,text):
    with open(name + '.html','w') as file:
        file.write('<!doctype html>' + text)
        return name + '.html'
    
write_html('demo_one', '<html><body>one</body></html>')
write_html('demo_two', '<html><body>two</body></html>')

'demo_two.html'

## T4 — stable two-key bubble sort (8 min)

`sort_cards(cards)` returns a **new** list sorted first by number ascending, then for equal numbers by `red`, `green`, `blue`. The input list must remain unchanged. Do not use built-in sorting.

Required test:

```python
cards = [[2, 'blue'], [1, 'blue'], [2, 'red'], [1, 'red'], [2, 'green'], [1, 'green']]
```

Expected return:

```python
[[1, 'red'], [1, 'green'], [1, 'blue'], [2, 'red'], [2, 'green'], [2, 'blue']]
```

Print the result and the unchanged original.

In [5]:
# T4 — YOUR ATTEMPT
# identifier: sort_cards
cards = [[2, 'blue'], [1, 'blue'], [2, 'red'], [1, 'red'], [2, 'green'], [1, 'green']]

def sort_cards(inp):
    cards = [_ for _ in inp]
    colours = ['red','green','blue']
    length = len(cards)
    swapped = True
    while swapped:
        swapped = False
        for n in range(length-1):
            curr = cards[n][0]
            nxt = cards[n+1][0]
            currcolour = colours.index(cards[n][1])
            nxtcolour = colours.index(cards[n+1][1])
            if curr == nxt and currcolour > nxtcolour:
                cards[n],cards[n+1] = cards[n+1],cards[n]
                swapped = True
            elif curr > nxt:
                cards[n],cards[n+1] = cards[n+1],cards[n]
                swapped = True
    return cards
    
print(cards)
print(sort_cards(cards))

[[2, 'blue'], [1, 'blue'], [2, 'red'], [1, 'red'], [2, 'green'], [1, 'green']]
[[1, 'red'], [1, 'green'], [1, 'blue'], [2, 'red'], [2, 'green'], [2, 'blue']]


## T5 — circular queue wrap probe (12 min)

`CircularQueue(size=4)` stores a fixed list, `head`, `tail` (next free slot), and `numberItems` (live count, initially 0).

- `enqueue(item)` returns `False` if full; otherwise inserts, wraps the tail, increments the count, and returns `True`.
- `dequeue()` returns `False` if empty; otherwise removes and returns the head item, wraps the head, and decrements the count.
- `items()` returns current contents from the head for exactly `numberItems` slots, without removal.

**Required test:** enqueue `a,b,c,d`; dequeue twice; enqueue `e,f`; print `items()` (must be `['c','d','e','f']`); dequeue five times (the fifth must be `False`).

In [6]:
# T5 — YOUR ATTEMPT
# identifier: CircularQueue

class CircularQueue:
    def __init__(self, size=4):
        self.__queue = [None for _ in range(size)]
        self.__size = size
        self.__head = 0
        self.__tail = 0
        self.__numberItems = 0
        
    def enqueue(self,item):
        if None not in self.__queue:
            self.__head = 0
            self.__tail = 0
            return False
        self.__queue[self.__tail] = item
        self.__tail += 1
        if self.__tail >= self.__size:
            self.__tail = 0
        self.__numberItems += 1
        return True
    
    def dequeue(self):
        if self.__queue is [None for _ in range(self.__size)]:
            self.__head = 0
            self.__tail = 0
            return False
        out = self.__queue[self.__head] 
        if out is None:
            return False
        self.__queue[self.__head] = None
        self.__head += 1
        if self.__head >= self.__size:
            self.__head = 0
        self.__numberItems -= 1
        return out
    
    def items(self):
        output = []
        start = self.__head
        for n in range(self.__numberItems):
            output.append(self.__queue[start])
            start += 1
            if start >= self.__size:
                start = 0
        return output
    
test = CircularQueue()
test.enqueue('a')
test.enqueue('b')
test.enqueue('c')
test.enqueue('d')
print(test.dequeue())
print(test.dequeue())
test.enqueue('e')
test.enqueue('f')
print(test.items())

a
b
['c', 'd', 'e', 'f']


## T6 — linked-list delete: head, missing, empty (10 min)

`Node(data)` stores private data and next; provide `getData`, `getNext`, `setNext`.

`LinkedList` starts empty:

- `append(data)` adds at the end.
- `delete(data)` returns `True` after deleting the first matching node; returns `False` on an empty list or a miss; never crashes.
- `items()` returns the data from head to tail.

**Required test:** append `5,7,9`; delete head `5`; delete missing `99`; print items; call `delete(1)` on a new empty list. Expected returns: `True`, `False`, list `[7,9]`, `False`.

In [7]:
# T6 — YOUR ATTEMPT
# identifiers: Node, LinkedList
class Node:
    def __init__(self, data = None):
        self.__data = data
        self.__next = None
        
    def getData(self):
        return self.__data
    
    def getNext(self):
        return self.__next
    
    def setNext(self, next):
        self.__next = next
        
class LinkedList:
    def __init__(self):
        self.__head = None
        
    def append(self, data):
        new = Node(data)
        #empty case
        if self.__head is None:
            self.__head = new
            return True
        #after head
        elif self.__head.getNext() is None:
            self.__head.setNext(new)
            return True
        #regular case
        previous = None
        current = self.__head
        while current:
            previous = current
            current = current.getNext()
        #current is none and previous is last node
        previous.setNext(new)
        return True
        
    def delete(self,data):
        #empty case
        if self.__head is None:
            return False
        #delete head
        elif data == self.__head.getData():
            self.__head = self.__head.getNext()
            return True
        #regular case
        previous = None
        current = self.__head
        while current and current.getData() != data:
            previous = current
            current = current.getNext()
        if current is None:
            return False
        #current is what we want to delete
        previous.setNext(current.getNext())
        return True
    
    def items(self):
        #empty case
        if self.__head is None:
            return None
        #traverse
        output = []
        current = self.__head
        while current:
            output.append(current.getData())
            current = current.getNext()
        return output
    
test = LinkedList()
new = LinkedList()
print(
test.append(5),
test.append(7),
test.append(9),
test.delete(5),
test.delete(99),
test.items(),
new.delete(1))

True True True True False [7, 9] False


---
# Checkpoints

Run this only after the six attempts. A FAIL means that rewrite does not yet have evidence. The checkpoint creates and removes `final_taper.db`; it leaves the two HTML files so you can inspect their sizes.

In [8]:
# CHECKPOINTS — FINAL TAPER
import os
import sqlite3


def check_t1():
    global hash_table
    hash_table = [None for _ in range(100)]
    read_data('records.txt')
    if find_record(475) != 'law':
        return False, f"475 returned {find_record(475)!r}, expected 'law'"
    if find_record(9999) is not False:
        return False, 'missing key must return False'
    return True, 'ok'


def check_t2():
    path = 'final_taper.db'
    if os.path.exists(path):
        os.remove(path)
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute('CREATE TABLE Villa(villaID INTEGER PRIMARY KEY, villaName TEXT)')
    conn.execute('CREATE TABLE Villa_Booking(villaID INTEGER, date TEXT)')
    conn.execute("INSERT INTO Villa VALUES (3, 'Dolphin')")
    conn.executemany('INSERT INTO Villa_Booking VALUES (3, ?)', [('10-Apr',), ('11-Apr',)])
    conn.commit()
    dates = requested_dates('Apr', 8, 4)
    got = availability(conn, 'Dolphin', 'Apr', 8, 4)
    conn.close()
    os.remove(path)
    if dates != ['08-Apr', '09-Apr', '10-Apr', '11-Apr']:
        return False, f'dates are {dates}'
    if got != (['08-Apr', '09-Apr'], ['10-Apr', '11-Apr']):
        return False, f'availability returned {got}'
    return True, 'ok'


def check_t3():
    a = write_html('demo_one', '<html><body>one</body></html>')
    b = write_html('demo_two', '<html><body>two</body></html>')
    if a != 'demo_one.html' or b != 'demo_two.html':
        return False, f'filenames returned {(a, b)}'
    if not os.path.exists(a) or not os.path.exists(b):
        return False, 'one or both files do not exist'
    if os.path.getsize(a) == 0 or os.path.getsize(b) == 0:
        return False, 'one or both files are empty'
    return True, 'ok'


def check_t4():
    cards = [[2, 'blue'], [1, 'blue'], [2, 'red'], [1, 'red'], [2, 'green'], [1, 'green']]
    original = [card[:] for card in cards]
    expected = [[1, 'red'], [1, 'green'], [1, 'blue'], [2, 'red'], [2, 'green'], [2, 'blue']]
    got = sort_cards(cards)
    if got != expected:
        return False, f'sort returned {got}'
    if cards != original:
        return False, 'input list was changed'
    return True, 'ok'


def check_t5():
    q = CircularQueue(4)
    if [q.enqueue(x) for x in 'abcd'] != [True, True, True, True]:
        return False, 'first four enqueues must succeed'
    if q.enqueue('overflow') is not False:
        return False, 'full enqueue must return False'
    if q.dequeue() != 'a' or q.dequeue() != 'b':
        return False, 'first dequeues are wrong'
    q.enqueue('e')
    q.enqueue('f')
    if q.items() != ['c', 'd', 'e', 'f']:
        return False, f'wrapped items are {q.items()}'
    got = [q.dequeue() for _ in range(5)]
    if got != ['c', 'd', 'e', 'f', False]:
        return False, f'dequeues are {got}'
    return True, 'ok'


def check_t6():
    linked = LinkedList()
    for x in (5, 7, 9):
        linked.append(x)
    if linked.delete(5) is not True or linked.items() != [7, 9]:
        return False, f'head delete left {linked.items()}'
    if linked.delete(99) is not False:
        return False, 'missing delete must return False'
    if LinkedList().delete(1) is not False:
        return False, 'empty delete must return False'
    return True, 'ok'


checks = [check_t1, check_t2, check_t3, check_t4, check_t5, check_t6]
for index, check in enumerate(checks, 1):
    try:
        ok, detail = check()
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    print(f"T{index}: {'PASS' if ok else 'FAIL — ' + detail}")

T1: PASS
T2: PASS
T3: PASS
T4: PASS
T5: PASS
T6: PASS
